# Whisper WebUI Persian - Colab (faster-whisper)

Runs the full WebUI (VAD, long audio, video, URLs, SRT/VTT/TXT/JSON) against the fine-tuned
Persian models on a free Colab GPU, using the **faster-whisper** backend.

faster-whisper only reads models in CTranslate2 format, so the Persian checkpoints are converted
once in step 4. The conversion takes about a minute and the result is reused for the rest of the
session.

**Before you start:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

Then run the cells in order.

## 1. Check the GPU

If this errors or prints nothing, the runtime is still on CPU - go back and switch it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 2. Check out the project

In [ ]:
REPO = "https://github.com/ashahdev403/Whisper-WebUi-Persian.git"

import os

if not os.path.isdir("Whisper-WebUi-Persian"):
    !git clone {REPO}

%cd /content/Whisper-WebUi-Persian
!git log --oneline -1
!ls

### Alternative: run from a local copy

If you have changes that are not pushed yet, zip the project folder locally, upload the zip with the
file browser on the left, and run this **instead** of the cell above:

```python
!unzip -q -o /content/Whisper-WebUi-Persian.zip -d /content/
%cd /content/Whisper-WebUi-Persian
```

## 3. Install dependencies

Deliberately **not** `pip install -r requirements-fasterWhisper.txt` - that pulls in
`requirements.txt`, which reinstalls torch and can break Colab's CUDA build. Colab already ships
torch, torchaudio, numpy and ffmpeg, so only the rest is installed here.

`ctranslate2` and `faster-whisper` are the backend itself; `transformers` is still needed for the
one-off model conversion.

Takes about a minute. Ignore pip dependency-resolver warnings about preinstalled Colab packages.

In [ ]:
!pip install -q ctranslate2 faster-whisper "transformers>=4.48.0" accelerate gradio json5 ffmpeg-python yt-dlp more-itertools altair intervaltree srt

!ffmpeg -version | head -1

import ctranslate2, faster_whisper
print("ctranslate2", ctranslate2.__version__, "| faster-whisper", faster_whisper.__version__)

# ctranslate2 finds CUDA independently of torch, and this is what decides which compute types are
# available. A GPU runtime that torch can see but ctranslate2 cannot would fail much later, at model
# load, with a confusing "float16 not supported" error.
CUDA_DEVICES = ctranslate2.get_cuda_device_count()
DEVICE = "cuda" if CUDA_DEVICES > 0 else "cpu"
print(f"\nctranslate2 sees {CUDA_DEVICES} CUDA device(s) -> running on {DEVICE.upper()}")
print("supported compute types:", sorted(ctranslate2.get_supported_compute_types(DEVICE)))

if DEVICE == "cpu":
    print("\n" + "!" * 78)
    print("NO GPU. Persian Large v3 on CPU is far too slow to be useful - expect many times")
    print("real time. Set the runtime to a GPU (Runtime -> Change runtime type -> T4 GPU) and")
    print("run the cells again. If Colab will not give you one, pick Persian Small in the UI.")
    print("!" * 78)

## 4. Convert the Persian models to CTranslate2

faster-whisper cannot read a HuggingFace checkpoint - it needs CTranslate2 format. Both Persian
models are converted once into `/content/ct2/` so they are both in the Model dropdown.

Expect a few minutes: Persian Large v3 is a ~3 GB download. It is the more accurate of the two;
Persian Small is roughly six times faster.

`float16` is the right quantization on a GPU. Use `int8` if you are short on VRAM - it is smaller
and faster, but noticeably less accurate.

In [ ]:
QUANTIZATION = "float16"   # "float16" on GPU, "int8" to save VRAM

import json, os, subprocess

SOURCES = {
    "Persian Small": "AmirMohseni/whisper-small-persian-bf16",
    "Persian Large v3": "AmirMohseni/whisper-large-v3-persian-bf16",
}

converted = {}

for name, repo in SOURCES.items():
    out_dir = "/content/ct2/" + name.replace(" ", "-").lower()

    # preprocessor_config.json, not model.bin, is what makes the conversion usable - see below.
    # Using it as the marker also re-does conversions left over from an earlier, incomplete run.
    complete = (os.path.exists(out_dir + "/model.bin")
                and os.path.exists(out_dir + "/preprocessor_config.json"))

    if complete:
        print(f"{name}: already converted, skipping")
    else:
        print(f"{name}: converting {repo} -> {out_dir} ({QUANTIZATION}) ...")
        subprocess.run([
            "ct2-transformers-converter", "--model", repo,
            "--output_dir", out_dir, "--quantization", QUANTIZATION, "--force",
            # By default the converter writes only config.json, model.bin and vocabulary.json, and
            # none of them record the number of mel bins. faster-whisper reads that from
            # preprocessor_config.json and silently falls back to 80 when the file is absent.
            # large-v3 is the one Whisper model that uses 128, so without this it fails with
            # "expected an input with shape (1, 128, 3000), but got (1, 80, 3000)".
            # Only this file is copied: --copy_files aborts the whole conversion if a name is
            # missing, and these repositories ship no tokenizer.json (faster-whisper fetches that
            # from the Hub by itself).
            "--copy_files", "preprocessor_config.json",
        ], check=True)

    converted[name] = out_dir

print("\nPersian models that will appear in the Model dropdown:")
for name, path in converted.items():
    size = sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path))
    with open(os.path.join(path, "preprocessor_config.json"), encoding="utf-8") as handle:
        mels = json.load(handle).get("feature_size", "?")
    print(f"  {name:20s} {size / 1e6:7.0f} MB  {mels} mel bins  {path}")

## 5. Write the Colab config

`app-shared.py` and friends are presets with no argument parsing - passing `--compute_type` to them
does nothing. Everything therefore goes through a config file, which `WHISPER_WEBUI_CONFIG` points
at. The repository's own `config.json5` is left untouched.

The stock Whisper names are listed too: faster-whisper downloads ready-made CTranslate2 builds for
those, so they need no conversion.

In [ ]:
import ctranslate2, json, os

CONFIG_PATH = "/content/config.colab.json5"

# Pick a compute type the device actually supports, rather than assuming float16. On CPU that type
# is unavailable and the model fails to load with "Requested float16 compute type, but the target
# device or backend do not support efficient float16 computation."
supported = ctranslate2.get_supported_compute_types(DEVICE)
COMPUTE_TYPE = next(t for t in ("float16", "int8_float32", "int8", "float32") if t in supported)

models = [{"name": name, "url": path, "language": "Persian"} for name, path in converted.items()]
models += [{"name": n, "url": n} for n in ["tiny", "base", "small", "medium", "large-v3"]]

config = {
    "models": models,
    "whisper_implementation": "faster-whisper",
    # Large v3 is the more accurate model, but on CPU it is unusable - default to Small there
    "default_model_name": "Persian Large v3" if DEVICE == "cuda" else "Persian Small",
    "compute_type": COMPUTE_TYPE,
    "language": "Persian",
    "task": "transcribe",
    "default_vad": "silero-vad",
    "vad_max_merge_size": 30,
    "input_audio_max_duration": -1,
    "share": True,
    "verbose": True,
    # Repetition loops are the main failure mode on long Persian audio; not feeding the previous
    # segment back in makes them far less likely
    "condition_on_previous_text": False,
}

with open(CONFIG_PATH, "w", encoding="utf-8") as handle:
    json.dump(config, handle, ensure_ascii=False, indent=4)

os.environ["WHISPER_WEBUI_CONFIG"] = CONFIG_PATH

from src.config import ApplicationConfig

loaded = ApplicationConfig.parse_file(CONFIG_PATH)
print("backend      :", loaded.whisper_implementation)
print("device       :", DEVICE)
print("compute type :", loaded.compute_type)
print("VAD          :", loaded.default_vad)
print()
print(f"The Model dropdown will contain these {len(loaded.models)} entries:")
for model in loaded.models:
    marker = "  <- default" if model.name == loaded.default_model_name else ""
    kind = "Persian, fine-tuned" if model.language == "Persian" else "stock multilingual"
    print(f"  {model.name:20s} {kind}{marker}")

assert loaded.default_model_name in loaded.get_model_names(), "default model is missing!"

## 6. Quick CLI test

Faster to debug than the UI, and it proves the whole chain: CT2 model load -> VAD -> transcription
-> subtitle files. If something is wrong, the traceback lands right here instead of inside a Gradio
worker thread.

Upload a Persian audio or video file with the file browser on the left and set `AUDIO` to its path.
Left as-is it generates a test tone, which transcribes to nothing - that only checks the pipeline
runs, not that it is any good.

In [ ]:
AUDIO = "/content/sample.wav"
MODEL = loaded.default_model_name   # same model the WebUI will open with; "Persian Small" is faster

import os

if not os.path.exists(AUDIO):
    print("No file at", AUDIO, "- generating a 40s test tone instead.")
    !ffmpeg -y -loglevel error -f lavfi -i "sine=frequency=440:duration=40" -ar 16000 -ac 1 {AUDIO}

# WHISPER_WEBUI_CONFIG carries the backend, compute type and model list
!WHISPER_WEBUI_CONFIG={CONFIG_PATH} python cli.py "{AUDIO}" --model "{MODEL}" --vad silero-vad --output_dir /content/out

In [ ]:
# Show what the CLI produced
import glob

for path in sorted(glob.glob("/content/out/*")):
    print("=" * 70)
    print(path)
    print("=" * 70)

    if path.endswith((".srt", ".txt", ".vtt")):
        with open(path, encoding="utf-8") as handle:
            print(handle.read()[:2000])

## 7. Launch the WebUI

This cell keeps running for as long as the server is up. Click the `https://xxxxx.gradio.live` link
that appears next to **Running on public URL**.

`app.py` is used rather than `app-shared.py` because the config already sets `share: true` - and
because the preset launchers ignore command line arguments, which makes them easy to misread.

Stop the cell to shut the server down.

In [ ]:
!WHISPER_WEBUI_CONFIG={CONFIG_PATH} python app.py

## Notes

- **Public link.** The config sets `share: true`, so Gradio publishes a link anyone with the URL can
  use, for 72 hours or until you stop the cell. Don't put sensitive audio through it. Set
  `"share": False` in step 5 and use the local URL if that matters.
- **Why the conversion.** faster-whisper reads CTranslate2 only. The `transformers` backend runs the
  same HuggingFace checkpoints with no conversion but is slower on GPU - switch by setting
  `"whisper_implementation": "transformers"` in step 5 and pointing the model URLs back at the
  `AmirMohseni/...` repositories.
- **Repetition loops.** `condition_on_previous_text` is off in this config. It is the single most
  effective setting against a segment collapsing into a repeated word on long Persian audio; the
  cost is slightly less coherent wording across segment boundaries.
- **Long files.** Keep VAD on `silero-vad`. There is no length limit in this notebook.
- **Sessions.** Colab reclaims idle runtimes. Runtime -> Manage sessions -> terminate when done,
  otherwise it eats your free compute.
- **Nothing persists.** `/content/ct2/` and the HuggingFace cache are lost when the runtime resets,
  so the conversion runs again next session. Mount Drive and convert into a Drive path to avoid it.